# Sprint 8 · Sesión única práctica · Proyecto de estadística aplicada con Python

**Proyecto:** NovaRetail+ · Exploración de relaciones y correlaciones  
**Duración:** 70 minutos  
**Modalidad:** práctica guiada en Google Colab / Jupyter Notebook  

En esta sesión vamos a construir un análisis estadístico corto, práctico y comunicable. El objetivo no es memorizar fórmulas, sino entender cómo usar la estadística para responder una pregunta de negocio con evidencia visual y numérica.


## Fecha

> Completa aquí la fecha de la sesión.


## Objetivo de la sesión

Al finalizar la sesión, el/la estudiante podrá:

1. Explicar de forma simple qué mide una correlación.
2. Diferenciar correlación de causalidad.
3. Cargar y explorar un dataset en Python usando `pandas`.
4. Limpiar datos mínimos antes de calcular relaciones estadísticas.
5. Crear gráficas sencillas con `.plot()`.
6. Crear gráficas agrupadas con `seaborn`.
7. Calcular correlaciones de Pearson y Spearman.
8. Construir un mini-reporte ejecutivo con hallazgos, evidencia y limitaciones.


## Agenda sugerida · 70 minutos

| Tiempo | Bloque | Actividad |
|---:|---|---|
| 0–5 min | Contexto del proyecto | Pregunta guía y entregable esperado |
| 5–15 min | Introducción matemática corta | Media, desviación, covarianza, correlación y causalidad |
| 15–25 min | Carga y exploración | Leer CSV, revisar estructura, tipos y nulos |
| 25–35 min | Limpieza mínima | Estandarizar categorías, imputar nulos, tratar outliers |
| 35–45 min | Visualización sencilla | Gráficas con `.plot()` |
| 45–58 min | Correlaciones y visualización agrupada | Pearson, Spearman, heatmap y `seaborn` con grupos |
| 58–65 min | Reporte ejecutivo | Hallazgos, implicaciones y límites |
| 65–70 min | Cierre | Validación de conocimiento, takeaways y canales de ayuda |


## Contexto del negocio

NovaRetail+ es una plataforma de e-commerce que quiere entender qué variables de comportamiento del cliente están más asociadas con el **ingreso generado**.

**Pregunta guía del proyecto:**

> ¿Qué factores de comportamiento parecen estar más relacionados con el ingreso generado por cliente?

Este análisis es **exploratorio y correlacional**. Sirve para generar hipótesis y priorizar análisis posteriores, pero no demuestra causalidad.


## Entregable de la clase

Al final de la sesión, cada estudiante debe tener:

1. Un notebook ejecutado con carga, exploración, limpieza, gráficas y correlaciones.
2. Tres hallazgos basados en evidencia.
3. Una recomendación de negocio.
4. Una advertencia explícita sobre causalidad y limitaciones.


---

# 1. Introducción matemática corta

La estadística nos ayuda a resumir datos y detectar patrones.

## 1.1 Media

La media resume el centro de una variable numérica:

$$
\bar{x} = \frac{x_1 + x_2 + \dots + x_n}{n}
$$

En negocio, por ejemplo, la media puede responder:

> ¿Cuál es el ingreso promedio generado por cliente?

## 1.2 Desviación

La desviación mide qué tan lejos está un dato de la media:

$$
x_i - \bar{x}
$$

Si muchos datos están lejos de la media, la variable tiene alta dispersión.

## 1.3 Covarianza

La covarianza mira si dos variables se mueven juntas:

$$
cov(x,y) = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{n-1}
$$

Si cuando `x` sube, `y` también tiende a subir, la covarianza suele ser positiva.

## 1.4 Correlación

La correlación estandariza la covarianza para dejarla entre -1 y 1:

$$
r = \frac{cov(x,y)}{s_x s_y}
$$

Lectura rápida:

| Valor aproximado | Interpretación |
|---:|---|
| Cerca de 1 | Relación positiva fuerte |
| Cerca de 0 | Relación débil o inexistente |
| Cerca de -1 | Relación negativa fuerte |

Regla crítica:

> Correlación no implica causalidad. Dos variables pueden moverse juntas por coincidencia, por una tercera variable o por sesgo en los datos.


## Mini-discusión inicial

Responde antes de ejecutar código:

1. Si `visitas_mes` e `ingreso_generado` tienen correlación positiva, ¿significa que aumentar visitas causa más ingresos?
2. ¿Qué otra variable podría explicar esa relación?
3. ¿Por qué un gráfico puede ser útil antes de calcular una correlación?


---

# 2. Carga del dataset

Trabajaremos con un dataset sintético de clientes de NovaRetail+.

El archivo de clase es:

` sprint08_proyecto_correlaciones_novaretail.csv `

Si estás en Google Colab, sube el CSV al panel lateral de archivos o usa la celda de carga manual si aparece un error.


In [ ]:
# Imports básicos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual simple
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Librerías cargadas correctamente.")


In [ ]:
# Cargar dataset
DATA_PATH = "sprint08_proyecto_correlaciones_novaretail.csv"

try:
    df = pd.read_csv(DATA_PATH)
    print("Dataset cargado desde archivo local:", DATA_PATH)
except FileNotFoundError:
    print("No se encontró el archivo local. Si estás en Google Colab, sube el CSV cuando se solicite.")
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
    df = pd.read_csv(DATA_PATH)
    print("Dataset cargado desde archivo subido:", DATA_PATH)

df.head()


In [ ]:
# Tamaño del dataset
print("Filas y columnas:", df.shape)

# Muestra aleatoria para evitar mirar siempre los mismos registros
df.sample(5, random_state=42)


## Diccionario rápido de variables

| Variable | Descripción |
|---|---|
| `cliente_id` | Identificador único del cliente |
| `region` | Región o ciudad principal |
| `dispositivo` | Tipo de dispositivo usado |
| `canal_adquisicion` | Canal por el que llegó el cliente |
| `segmento` | Tipo de cliente |
| `edad` | Edad del cliente |
| `meses_antiguedad` | Meses desde su primera interacción |
| `visitas_mes` | Visitas al sitio/app en el mes |
| `compras_mes` | Número de compras en el mes |
| `tiempo_app_min` | Tiempo mensual estimado en app/web |
| `gasto_marketing` | Inversión de marketing atribuida al cliente |
| `satisfaccion` | Score de satisfacción de 1 a 5 |
| `usa_cupon` | 1 si usó cupón, 0 si no |
| `abandono` | 1 si el cliente abandonó, 0 si sigue activo |
| `ticket_promedio` | Valor promedio por compra |
| `ingreso_generado` | Métrica foco del análisis |


---

# 3. Exploración inicial

Antes de calcular estadística, revisamos:

- tipos de datos,
- valores nulos,
- variables numéricas,
- variables categóricas,
- posibles inconsistencias.


In [ ]:
# Tipos de datos
df.info()


In [ ]:
# Valores faltantes por columna
missing = (
    df.isna().sum()
    .reset_index()
    .rename(columns={"index": "columna", 0: "nulos"})
)

missing["porcentaje"] = (missing["nulos"] / len(df) * 100).round(2)
missing.sort_values("nulos", ascending=False)


In [ ]:
# Estadística descriptiva de variables numéricas
df.describe()


In [ ]:
# Revisión de valores únicos en variables categóricas principales
cat_cols = ["region", "dispositivo", "canal_adquisicion", "segmento"]

for col in cat_cols:
    print("\n", col)
    print(df[col].value_counts(dropna=False).head(10))


## Preguntas guiadas

1. ¿Qué columnas tienen valores faltantes?
2. ¿Qué variables parecen numéricas?
3. ¿Hay categorías escritas de forma inconsistente?
4. ¿Cuál es la variable objetivo del proyecto?


---

# 4. Limpieza mínima para análisis estadístico

La limpieza será deliberadamente simple. No vamos a “perfeccionar” el dataset; solo haremos lo necesario para que el análisis sea más confiable.

Decisiones:

1. Estandarizar texto en variables categóricas.
2. Reemplazar nulos numéricos con mediana.
3. Reemplazar nulos categóricos con `sin_region`.
4. Crear una versión tratada de outliers para comparar.


In [ ]:
# Crear copia de trabajo
df_clean = df.copy()

# 1) Estandarizar texto en columnas categóricas
for col in ["region", "dispositivo", "canal_adquisicion", "segmento"]:
    df_clean[col] = (
        df_clean[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

# 2) Reemplazar región nula
df_clean["region"] = df_clean["region"].fillna("sin_region")

# 3) Reemplazar nulos numéricos con mediana
num_cols_with_na = ["edad", "satisfaccion", "gasto_marketing"]

for col in num_cols_with_na:
    median_value = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_value)
    print(f"{col}: nulos reemplazados con mediana = {median_value:,.2f}")

# 4) Verificar
df_clean.isna().sum()


In [ ]:
# Tratamiento simple de outliers: winsorizar ingreso y gasto al percentil 99
df_clean["ingreso_generado_w"] = df_clean["ingreso_generado"].clip(
    upper=df_clean["ingreso_generado"].quantile(0.99)
)

df_clean["gasto_marketing_w"] = df_clean["gasto_marketing"].clip(
    upper=df_clean["gasto_marketing"].quantile(0.99)
)

# Variable derivada: ingreso por compra
df_clean["ingreso_por_compra"] = np.where(
    df_clean["compras_mes"] > 0,
    df_clean["ingreso_generado_w"] / df_clean["compras_mes"],
    0
)

df_clean[["ingreso_generado", "ingreso_generado_w", "gasto_marketing", "gasto_marketing_w", "ingreso_por_compra"]].describe()


## Nota didáctica

Winsorizar no elimina registros. Solo limita valores extremos para que no dominen algunas métricas o gráficos.

No siempre se debe hacer. Depende del caso de negocio:

- Si un outlier es error de captura, se puede corregir o excluir.
- Si un outlier es real, puede ser el cliente más valioso.
- Si no estás seguro, documenta la decisión.


---

# 5. Visualizaciones sencillas con `.plot()`

Primero usaremos métodos nativos de pandas. Son rápidos y suficientes para una primera exploración.


In [ ]:
# Histograma sencillo de la métrica foco
df_clean["ingreso_generado_w"].plot(
    kind="hist",
    bins=30,
    title="Distribución del ingreso generado"
)

plt.xlabel("Ingreso generado")
plt.show()


In [ ]:
# Ingreso promedio por segmento
ingreso_por_segmento = (
    df_clean.groupby("segmento")["ingreso_generado_w"]
    .mean()
    .sort_values(ascending=False)
)

ingreso_por_segmento.plot(
    kind="bar",
    title="Ingreso promedio por segmento"
)

plt.ylabel("Ingreso promedio")
plt.xlabel("Segmento")
plt.xticks(rotation=0)
plt.show()


In [ ]:
# Scatter simple con pandas
df_clean.plot(
    kind="scatter",
    x="visitas_mes",
    y="ingreso_generado_w",
    title="Visitas mensuales vs ingreso generado"
)

plt.xlabel("Visitas mensuales")
plt.ylabel("Ingreso generado")
plt.show()


## Preguntas rápidas

1. ¿La distribución del ingreso es simétrica o sesgada?
2. ¿Qué segmento tiene mayor ingreso promedio?
3. ¿El scatter sugiere una relación positiva, negativa o débil?


---

# 6. Gráficas agrupadas con Seaborn

Ahora usamos `seaborn` para visualizar grupos. Esto ayuda a ver si una relación cambia según segmento, región o canal.


In [ ]:
# Scatter agrupado por segmento
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df_clean.sample(800, random_state=42),
    x="visitas_mes",
    y="ingreso_generado_w",
    hue="segmento",
    alpha=0.65
)

plt.title("Visitas vs ingreso generado, agrupado por segmento")
plt.xlabel("Visitas mensuales")
plt.ylabel("Ingreso generado")
plt.show()


In [ ]:
# Boxplot agrupado por segmento
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=df_clean,
    x="segmento",
    y="ingreso_generado_w"
)

plt.title("Distribución de ingreso generado por segmento")
plt.xlabel("Segmento")
plt.ylabel("Ingreso generado")
plt.show()


In [ ]:
# Barras agrupadas: ingreso promedio por región y segmento
tabla_region_segmento = (
    df_clean.groupby(["region", "segmento"], as_index=False)["ingreso_generado_w"]
    .mean()
)

plt.figure(figsize=(11, 5))

sns.barplot(
    data=tabla_region_segmento,
    x="region",
    y="ingreso_generado_w",
    hue="segmento"
)

plt.title("Ingreso promedio por región y segmento")
plt.xlabel("Región")
plt.ylabel("Ingreso promedio")
plt.xticks(rotation=20)
plt.show()


## Lectura guiada

Las gráficas agrupadas permiten detectar algo importante:

> El promedio general puede ocultar comportamientos por grupo.

Por ejemplo, una relación puede verse fuerte en el total, pero más débil dentro de cada segmento. Esa es una razón por la que no conviene confiar solo en una correlación agregada.


---

# 7. Correlaciones

Usaremos dos correlaciones principales:

## Pearson

Mide relación **lineal** entre dos variables numéricas.

- Útil cuando el patrón se parece a una línea.
- Más sensible a outliers.

## Spearman

Mide relación **monótona** usando rangos.

- Útil cuando la relación no es perfectamente lineal.
- Menos sensible a valores extremos.


In [ ]:
# Variables numéricas principales
num_cols = [
    "edad",
    "meses_antiguedad",
    "visitas_mes",
    "compras_mes",
    "tiempo_app_min",
    "gasto_marketing_w",
    "satisfaccion",
    "usa_cupon",
    "abandono",
    "ticket_promedio",
    "ingreso_generado_w"
]

df_clean[num_cols].head()


In [ ]:
# Correlación Pearson con la variable foco
pearson_target = (
    df_clean[num_cols]
    .corr(method="pearson")["ingreso_generado_w"]
    .drop("ingreso_generado_w")
    .sort_values(key=abs, ascending=False)
)

pearson_target


In [ ]:
# Correlación Spearman con la variable foco
spearman_target = (
    df_clean[num_cols]
    .corr(method="spearman")["ingreso_generado_w"]
    .drop("ingreso_generado_w")
    .sort_values(key=abs, ascending=False)
)

spearman_target


In [ ]:
# Comparación Pearson vs Spearman
corr_compare = pd.DataFrame({
    "pearson": pearson_target,
    "spearman": spearman_target
}).sort_values("spearman", key=abs, ascending=False)

corr_compare


In [ ]:
# Heatmap de correlaciones
plt.figure(figsize=(11, 7))

corr_matrix = df_clean[num_cols].corr(method="spearman")

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Mapa de calor de correlaciones Spearman")
plt.show()


## Cómo interpretar la magnitud

Usa esta regla solo como guía inicial:

| Magnitud absoluta | Lectura aproximada |
|---:|---|
| 0.00–0.10 | Muy débil |
| 0.10–0.30 | Débil |
| 0.30–0.50 | Moderada |
| 0.50–0.70 | Fuerte |
| 0.70–1.00 | Muy fuerte |

La interpretación final depende del contexto. En negocios, una correlación débil puede ser útil si afecta a muchos clientes o a mucho dinero.


---

# 8. Mini-reporte ejecutivo

Ahora convertimos el análisis en una respuesta de negocio.

No reportes solo números. Reporta:

1. Hallazgo.
2. Evidencia.
3. Interpretación.
4. Limitación.
5. Siguiente acción.


In [ ]:
# Top 5 variables más asociadas con ingreso generado
top_corr = corr_compare.reindex(corr_compare["spearman"].abs().sort_values(ascending=False).index).head(5)
top_corr


In [ ]:
# Generar un borrador automático de hallazgos
print("Mini-reporte ejecutivo · NovaRetail+\n")
print("Pregunta guía:")
print("¿Qué factores parecen estar más relacionados con el ingreso generado por cliente?\n")

print("Hallazgos principales:")
for variable, row in top_corr.iterrows():
    print(f"- {variable}: Spearman = {row['spearman']:.2f}, Pearson = {row['pearson']:.2f}")

print("\nInterpretación responsable:")
print("- Estas asociaciones muestran patrones, no causalidad.")
print("- Los resultados pueden estar afectados por segmento, canal, región, promociones o estacionalidad.")
print("- El siguiente paso sería validar hipótesis con segmentación adicional o experimentos controlados.")


## Plantilla para completar en clase

### Hallazgo 1

- Variable:
- Evidencia visual:
- Evidencia numérica:
- Interpretación:

### Hallazgo 2

- Variable:
- Evidencia visual:
- Evidencia numérica:
- Interpretación:

### Recomendación de negocio

- Acción sugerida:
- Segmento o grupo prioritario:
- Métrica que debería monitorearse:

### Limitación

- ¿Por qué esto no prueba causalidad?
- ¿Qué variable adicional te gustaría tener?


---

# 9. Preguntas de validación de conocimiento

Responde individualmente o en parejas.

1. ¿Qué significa que una correlación sea positiva?
2. ¿Qué diferencia hay entre Pearson y Spearman?
3. ¿Por qué un outlier puede afectar una correlación?
4. ¿Por qué no basta con decir “la correlación es alta”?
5. ¿Qué significa que correlación no implica causalidad?
6. ¿Por qué es útil agrupar visualizaciones por segmento, región o canal?
7. ¿Qué debería contener un mini-reporte estadístico para negocio?

## Respuestas esperadas / criterios

1. Que cuando una variable aumenta, la otra tiende a aumentar.
2. Pearson mide relación lineal; Spearman mide relación monótona basada en rangos.
3. Porque puede mover artificialmente la línea o la fuerza de la relación.
4. Porque se necesita contexto, visualización, tamaño del efecto, muestra, sesgos y objetivo.
5. Que una asociación estadística no demuestra que una variable cause la otra.
6. Porque los patrones pueden cambiar por grupo y el promedio general puede ocultar diferencias.
7. Contexto, método, hallazgos, evidencia, interpretación, limitaciones y próximos pasos.


---

# 10. Takeaways

- La correlación resume la relación entre variables, pero no reemplaza el análisis visual.
- Pearson y Spearman responden preguntas distintas.
- La limpieza mínima es necesaria antes de calcular estadísticas.
- Las gráficas con `.plot()` son suficientes para una primera lectura rápida.
- `seaborn` permite comparar grupos de forma más clara.
- Un buen análisis estadístico no termina en el número: termina en una interpretación responsable.
- Correlación no implica causalidad; debe reportarse siempre como limitación.


---

# 11. Canales de ayuda

Recuerda los canales de ayuda disponibles:

- `DATA-CO-LEARNING`: revisa los horarios de atención en la guía del estudiante. Hay espacios de apoyo para dudas puntuales.
- `DA_CONSULTA`: publica tus preguntas sobre contenido de la plataforma o proyecto. Agrega el tag `@Dataconsulta`.
- `SPRINT FOCUS`: sesiones extra para profundizar temas de cada sprint.
- `SESIONES 1:1`: agenda sesiones con tutor para resolver dudas de proyecto, curso, carrera o problemas técnicos.
- `CANAL DE DISCORD DE COMMUNITY`: comparte recursos, preguntas o hallazgos con tu cohorte.


## Cierre y próximos pasos

Antes de cerrar, verifica que tu notebook tenga:

- Dataset cargado correctamente.
- Limpieza documentada.
- Al menos 3 gráficas.
- Tabla de correlaciones.
- Mini-reporte con 2 o 3 hallazgos.
- Una limitación explícita sobre causalidad.

Para seguir practicando, cambia la variable objetivo y repite el análisis. Por ejemplo, analiza qué se relaciona más con `abandono`, `compras_mes` o `satisfaccion`.
